# `panorama.prismas` — Demo

Worked examples of the **`prismas`** module of PANORAMA: selection of the model atmospheres of a grid for a PRISMAS run, grid check of SPAMMS (grid entries needed and their model atmospheres), and renaming of the archives produced by PRISMAS in HPC mode.

* **Examples 1-3** — model selection in the catalogs distributed with PANORAMA: TLUSTY models selected by ranges and lists of values (names written to a file for an HTCondor submission), Kurucz-Castelli models with alpha enhancement and Kurucz-Mészáros models with quarter-step abundances and an open range in [C/M], and the checks of the selection criteria.
* **Example 4** — copy of the selected models from a *mock* local grid (empty files): only the complete OSTAR2002 model is copied, while a BSTAR2006 model with only one of its three files (also present in another subdirectory) and a model without files are reported in a single warning each and not copied.
* **Example 5** — SPAMMS grid check of the input file `input_SPAMMS.txt`, run only if the environment variable `SPAMMS_DIR` gives the folder where SPAMMS is installed.
* **Example 6** — renaming of *mock* TLUSTY archives of HPC mode: a dry run and the renaming with the inner folder and `.npy` files renamed too (including a truncated archive, two archives with the same target, an archive already renamed and a file that is not a model archive).

**Files:** the SPAMMS input file is read from `data_and_results/prismas/data/`, and the outputs (list of selected models, copied model files, mock and renamed archives and, with SPAMMS, the files and figure of the grid check) are written to `data_and_results/prismas/results/` (relative to `Demos/`, the folder of this notebook). The mock model files and archives are rebuilt at every run.

**SPAMMS:** example 5 needs a local installation of SPAMMS, read from the environment variable `SPAMMS_DIR` (e.g. `export SPAMMS_DIR=/home/user/SPAMMS` before launching Jupyter); it is skipped if the variable is not set. No internet access is needed.

Run the cells in order (*Run All*).

## Imports

Python packages (figures shown inline) and PANORAMA functions.

In [1]:
%matplotlib inline
import gzip
import io
from IPython.display import display
import numpy as np
import os
from pathlib import Path
import shutil
import tarfile
import warnings

from panorama import (
                      find_models,  # MODELS
                      needed_models,  # SPAMMS
                      rename_tars,  # ARCHIVES
                      savefig_pkl,  # MATPLOTLIB (tex module)
                     )

	PANORAMA v1.2.0 loaded...
		- To see the available functions, run:
			panorama.functions()          # Lists all functions, grouped by submodule
			panorama.functions('obs')     # Lists the functions of a single submodule
		- To see the docstring of a specific function (e.g. 'visibility'), run:
			help(panorama.visibility)
		- To silence this message, set the environment variable PANORAMA_QUIET=1



## Setup

Working directories, SPAMMS installation (optional) and folders of the mock model files and archives.

In [2]:
work_dir_data    = Path("./data_and_results/prismas/data/")
work_dir_results = Path("./data_and_results/prismas/results/")
work_dir_results.mkdir(parents=True, exist_ok=True)

SPAMMS_DIR  = os.environ.get("SPAMMS_DIR")  # Folder where SPAMMS is installed, from the environment variable SPAMMS_DIR (e.g. "/home/user/SPAMMS/"); None skips the SPAMMS grid check of example 5
SPAMMS_GRID = "T"  # Model grid of the SPAMMS input files: 'KC', 'KM' or 'T'

spamms_input = work_dir_data / "input_SPAMMS.txt"  # SPAMMS input file of example 5 (distributed with the repository; run with 'spamms.py')

# Mock model files (example 4) and HPC archives (example 6), written to 'results/' and rebuilt at every run so that old results never alter the outputs
mock_models_dir = work_dir_results / "mock_models"
selected_dir    = work_dir_results / "selected_models"
hpc_dir         = work_dir_results / "hpc_archives"
renamed_dir     = work_dir_results / "renamed_archives"
for folder in (mock_models_dir, selected_dir, hpc_dir, renamed_dir):
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)

## Example 1: TLUSTY models of a PRISMAS run

Selection by ranges and lists of values, with the model names written to a file (e.g. for an HTCondor submission).

In [3]:
selection_file = work_dir_results / "T_selection.dat"

models_T = find_models(
                       "T",
                       teff=(25000, 35000),  # Closed range [min, max] in K
                       logg=[3.5, 4.0],  # Discrete values
                       met=1,  # Solar composition (Z / Zsun)
                       output_file=selection_file
                      )

subgrids = sorted({m['subgrid'] for m in models_T})
print(f"\tModels per subgrid : { {s: sum(m['subgrid'] == s for m in models_T) for s in subgrids} }")
print(f"\tFirst lines of {selection_file.name}: {selection_file.read_text(encoding='utf-8').split()[:3]}")
print("\t(BSTAR2006 is used up to 27000 K and OSTAR2002 from 27500 K)")

models_V = find_models("T", teff=30000, logg=4.0, met=(0.03, 0.04), verbose=False)  # Fractional Z / Zsun selected with a range
print(f"\nZ / Zsun in [0.03, 0.04] at 30000 K, log g = 4.0: {[(m['model'], m['met']) for m in models_V]}")


14 T model(s) selected from the TLUSTY OSTAR2002/BSTAR2006 catalog
     Teff  logg  Z/Zsun  alpha      C  model
  ------------------------------------------
    25000  3.50   1.000      -      -  BG25000g350v2
    25000  4.00   1.000      -      -  BG25000g400v2
    26000  3.50   1.000      -      -  BG26000g350v2
    26000  4.00   1.000      -      -  BG26000g400v2
    27000  3.50   1.000      -      -  BG27000g350v2
    27000  4.00   1.000      -      -  BG27000g400v2
    27500  3.50   1.000      -      -  G27500g350v10
    27500  4.00   1.000      -      -  G27500g400v10
    30000  3.50   1.000      -      -  G30000g350v10
    30000  4.00   1.000      -      -  G30000g400v10
    32500  3.50   1.000      -      -  G32500g350v10
    32500  4.00   1.000      -      -  G32500g400v10
    35000  3.50   1.000      -      -  G35000g350v10
    35000  4.00   1.000      -      -  G35000g400v10

  Model names written to : data_and_results/prismas/results/T_selection.dat
	Models per subgrid : {

## Example 2: Kurucz-Castelli and Kurucz-Mészáros models

Alpha-enhanced Kurucz-Castelli models, and Kurucz-Mészáros models with a quarter-step metallicity and an open range in [C/M] (the first request of the 'KM' catalog takes a few seconds).

In [4]:
models_KC = find_models(
                        "KC",
                        teff=15000,
                        logg=(3.0, 4.0),
                        met=(-0.5, 0.0),
                        alpha=0.4  # AODFNEW models ([alpha/M] = +0.4)
                       )

models_KM = find_models(
                        "KM",
                        teff=5000,
                        logg=2.5,
                        met=-0.25,  # Quarter step: abundance code 'm03'
                        alpha=0.0,
                        carbon=(None, 0.0),  # Open range: [C/M] <= 0
                        verbose=False
                       )

print(f"\nKurucz-Meszaros models: {len(models_KM)}")
print(f"\t{'Model':<28} {'[M/H]':>6} {'[a/M]':>6} {'[C/M]':>6}")
for m in models_KM:
    print(f"\t{m['model']:<28} {m['met']:>6.2f} {m['alpha']:>6.2f} {m['carbon']:>6.2f}")


6 KC model(s) selected from the Kurucz-Castelli ATLAS9 ODFNEW catalog
     Teff  logg   [M/H]  alpha      C  model
  ------------------------------------------
    15000  3.00  -0.500   0.40      -  m0.50t15000g3.0am05ak2odfnew
    15000  3.00   0.000   0.40      -  p0.00t15000g3.0ap00ak2odfnew
    15000  3.50  -0.500   0.40      -  m0.50t15000g3.5am05ak2odfnew
    15000  3.50   0.000   0.40      -  p0.00t15000g3.5ap00ak2odfnew
    15000  4.00  -0.500   0.40      -  m0.50t15000g4.0am05ak2odfnew
    15000  4.00   0.000   0.40      -  p0.00t15000g4.0ap00ak2odfnew



Kurucz-Meszaros models: 7
	Model                         [M/H]  [a/M]  [C/M]
	amm03cm15op00t5000g25v20      -0.25   0.00  -1.50
	amm03cm13op00t5000g25v20      -0.25   0.00  -1.25
	amm03cm10op00t5000g25v20      -0.25   0.00  -1.00
	amm03cm08op00t5000g25v20      -0.25   0.00  -0.75
	amm03cm05op00t5000g25v20      -0.25   0.00  -0.50
	amm03cm03op00t5000g25v20      -0.25   0.00  -0.25
	amm03cp00op00t5000g25v20      -0.25   0.00   0.00


## Example 3: Checks of the selection criteria

Criteria not defined for the grid, empty or inverted ranges, unphysical values, unknown grids and an output directory without model directory.

In [5]:
checks = [
          ("[alpha/M] for TLUSTY", "T", {'alpha' : 0.0}),
          ("Inverted Teff range", "T", {'teff' : (35000, 25000)}),
          ("Negative Z / Zsun", "T", {'met' : -0.5}),
          ("Range without limits", "KC", {'logg' : (None, None)}),
          ("Unknown grid", "ATLAS12", {}),
          ("Copy without 'modeldir'", "T", {'output_dir' : selected_dir}),
         ]

print("Warnings are expected for the rejected cases below (TypeError or ValueError):")
for label, grid, criteria in checks:
    try:
        find_models(grid, **criteria, verbose=False)
        print(f"\t{label:<24} accepted")
    except (TypeError, ValueError) as e:
        print(f"\t{label:<24} rejected: {e}")

Warnings are expected for the rejected cases below (TypeError or ValueError):
	[alpha/M] for TLUSTY     rejected: Selection criterion 'alpha' must be None for the 'T' grid (TLUSTY OSTAR2002/BSTAR2006), which defines only 'teff', 'logg', 'met'. Got 0.0.
	Inverted Teff range      rejected: Effective temperature range 'teff' must satisfy min <= max. Got (35000.0, 25000.0).
	Negative Z / Zsun        rejected: Metallicity 'met' must be non-negative. Got -0.5.
	Range without limits     rejected: Surface gravity range 'logg' must have at least one limit (use None for no filter). Got (None, None).
	Unknown grid             rejected: Model grid 'grid' must be one of 'KC', 'KM', 'T' (case-insensitive). Got 'ATLAS12'.
	Copy without 'modeldir'  rejected: Output directory 'output_dir' must be given together with the model directory 'modeldir' (files are copied from it). Got modeldir=None.


## Example 4: Copy the selected models from a local grid

Model files located in the subdirectories of a mock grid; only the complete models are copied to a working directory, and incomplete, missing and repeated files are reported.

In [6]:
# Mock local grid (empty files): complete OSTAR2002 model, BSTAR2006 model with only its '.7' file (also in another subdirectory), and no file for G27500g400v10
mock_files = [
              mock_models_dir / "OSTAR2002" / "G30000g400v10.7",
              mock_models_dir / "OSTAR2002" / "G30000g400v10.5",
              mock_models_dir / "OSTAR2002" / "G30000g400v10.nst",
              mock_models_dir / "BSTAR2006" / "BG25000g400v2.7",
              mock_models_dir / "old_copies" / "BG25000g400v2.7",
             ]
for path in mock_files:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("Mock TLUSTY model file\n", encoding="utf-8")


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    models_copy = find_models(
                              "T",
                              teff=[25000, 27500, 30000],
                              logg=4.0,
                              met=1,
                              modeldir=mock_models_dir,
                              recursive=True,  # Subdirectories visited in alphabetical order
                              output_dir=selected_dir
                             )

print(f"\n\tFiles copied to {selected_dir.name}: {sorted(p.name for p in selected_dir.iterdir())}")
for m in models_copy:
    print(f"\t\t{m['model']:<15} {len(m['files'])} of 3 file(s) found")
print("\tWarnings")
for w in caught:
    print(f"\t\tWarning: {w.message}")


3 T model(s) selected from the TLUSTY OSTAR2002/BSTAR2006 catalog
     Teff  logg  Z/Zsun  alpha      C  model          found in 'modeldir'
  -----------------------------------------------------------------------
    25000  4.00   1.000      -      -  BG25000g400v2  BSTAR2006/BG25000g400v2.7
    27500  4.00   1.000      -      -  G27500g400v10  -
    30000  4.00   1.000      -      -  G30000g400v10  OSTAR2002/G30000g400v10.7, .5, .nst
  1 complete model(s) (3 file(s)) copied to : data_and_results/prismas/results/selected_models

	Files copied to selected_models: ['G30000g400v10.5', 'G30000g400v10.7', 'G30000g400v10.nst']
		BG25000g400v2   1 of 3 file(s) found
		G27500g400v10   0 of 3 file(s) found
		G30000g400v10   3 of 3 file(s) found
	Warnings


## Example 5: SPAMMS grid check

Grid entries needed by a SPAMMS input file, their model atmospheres and the figure of the needed entries on the grid. Run only if the environment variable `SPAMMS_DIR` is set (read in the setup cell).

In [7]:
print(f"\nSPAMMS input file: {spamms_input.name}")

if SPAMMS_DIR is None:
    print("\n\tSkipped: set the environment variable SPAMMS_DIR to the folder where SPAMMS is installed to run the grid check ('spamms.py -c').")
else:
    try:
        print()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            needed = needed_models(
                                   spamms_inputs=spamms_input,
                                   spamms_dir=SPAMMS_DIR,
                                   grid=SPAMMS_GRID,
                                   output_name=work_dir_results / "spamms_demo",
                                   show=False
                                  )
        for w in caught:
            print(f"\tWarning: {w.message}")

        print(f"\n\tGrid entries needed: {len(needed['needed'])}")
        print(f"\tModel atmospheres needed (first 5): {[m['model'] for m in needed['models']][:5]}")
        print(f"\tOutside the grid: {len(needed['not_in_grid'])}, dummy: {len(needed['dummy'])}, invalid: {len(needed['invalid'])}")
        print(f"\tOutput files: {needed['needed_file'].name}, {needed['missing_file'].name}")
        if needed['fig'] is not None:
            display(needed['fig'])  # Inline display (the figure is closed by 'savefig_pkl')
            written_files = savefig_pkl(needed['fig'], work_dir_results / f"spamms_demo_{SPAMMS_GRID}_grid")
            print(f"\tFigure saved as: {[p.name for p in written_files]}")
    except Exception as e:  # SPAMMS or the input file not found, or the SPAMMS run failed
        print(f"\n\tSkipped: {str(e).splitlines()[0]}")

print("\nInput check:")
try:
    needed_models([], SPAMMS_DIR or ".", SPAMMS_GRID)
except ValueError as e:  # At least one input file is needed
    print(f"\tRejected: {e}")


SPAMMS input file: input_SPAMMS.txt

	Skipped: set the environment variable SPAMMS_DIR to the folder where SPAMMS is installed to run the grid check ('spamms.py -c').

Input check:
	Rejected: SPAMMS inputs 'spamms_inputs' must contain at least one input file. Got an empty list.


## Example 6: Rename the archives of HPC mode

Mock TLUSTY archives of HPC mode (folder and five '.npy' files each), first checked with a dry run and then renamed to the names of modes 0 / 1, with their content renamed too.

In [8]:
# Mock archives: three valid ones, two with the same target (OSTAR2002 and BSTAR2006 models with the same Teff, log g and Z), a truncated one,
# one already renamed and a file that is not a model archive
spectrum_arrays = ("wave", "flux", "cont", "norm", "info")
archive_names   = ["G30000g400v10", "BG25000g400v2", "BG20000g300v2", "G30000g300v10", "BG30000g300v2", "C35000g400v10", "T27500_G4.00_Z1.000_V10"]
for name in archive_names:
    tar_buffer = io.BytesIO()
    with tarfile.open(fileobj=tar_buffer, mode="w") as tar:
        folder      = tarfile.TarInfo(name)
        folder.type = tarfile.DIRTYPE
        tar.addfile(folder)
        for array_name in spectrum_arrays:
            npy_buffer = io.BytesIO()
            np.save(npy_buffer, np.linspace(4000., 4100., 1001))  # Mock array
            member      = tarfile.TarInfo(f"{name}/{name}_{array_name}.npy")
            member.size = npy_buffer.getbuffer().nbytes
            npy_buffer.seek(0)
            tar.addfile(member, npy_buffer)
    archive_bytes = gzip.compress(tar_buffer.getvalue(), compresslevel=1)
    if name == "C35000g400v10":
        archive_bytes = archive_bytes[:len(archive_bytes) // 2]  # Truncated archive (e.g. a killed HTCondor job)
    (hpc_dir / f"{name}.tar.gz").write_bytes(archive_bytes)
(hpc_dir / "observing_log.tar.gz").write_bytes(gzip.compress(b"Not a model archive\n"))


print("\nDRY RUN:")
archives_dry = rename_tars(hpc_dir, "T", vmic=10, dry_run=True)

print("\nRENAMING CONTENT:")
archives = rename_tars(
                       hpc_dir,
                       "T",
                       vmic=10,  # Microturbulence of the synthesis ('--vmic' of prismas.py), in km/s
                       output_dir=renamed_dir,
                       keep_original=True,
                       ncores=2
                      )

renamed = [a for a in archives if a['status'] == "renamed"]
with tarfile.open(renamed[0]['target']) as tar:
    members = tar.getnames()
print(f"\n\tContent of {renamed[0]['target'].name}:")
for member in members:
    print(f"\t\t{member}")


DRY RUN:

7 T archive(s) found in data_and_results/prismas/results/hpc_archives (vmic = 10 km/s)
     Teff  logg  Z/Zsun  alpha      C  status    source -> target
  -----------------------------------------------------------------------------------------------
    20000  3.00   1.000      -      -  dry_run   BG20000g300v2.tar.gz -> T20000_G3.00_Z1.000_V10.tar.gz
    25000  4.00   1.000      -      -  dry_run   BG25000g400v2.tar.gz -> T25000_G4.00_Z1.000_V10.tar.gz
    27500  4.00   1.000      -      -  done      T27500_G4.00_Z1.000_V10.tar.gz -> T27500_G4.00_Z1.000_V10.tar.gz
    30000  3.00   1.000      -      -  dry_run   BG30000g300v2.tar.gz -> T30000_G3.00_Z1.000_V10.tar.gz
    30000  3.00   1.000      -      -  skipped   G30000g300v10.tar.gz -> T30000_G3.00_Z1.000_V10.tar.gz  [same target as BG30000g300v2.tar.gz]
    30000  4.00   1.000      -      -  dry_run   G30000g400v10.tar.gz -> T30000_G4.00_Z1.000_V10.tar.gz
    35000  4.00   2.000      -      -  dry_run   C35000g400v10.ta


Renaming archives:   0%|          | 0/5 [00:00<?, ?archive/s]


Renaming archives: 100%|██████████| 5/5 [00:00<00:00, 261.36archive/s]


7 T archive(s) found in data_and_results/prismas/results/hpc_archives (vmic = 10 km/s)
     Teff  logg  Z/Zsun  alpha      C  status    source -> target
  -----------------------------------------------------------------------------------------------
    20000  3.00   1.000      -      -  renamed   BG20000g300v2.tar.gz -> T20000_G3.00_Z1.000_V10.tar.gz
    25000  4.00   1.000      -      -  renamed   BG25000g400v2.tar.gz -> T25000_G4.00_Z1.000_V10.tar.gz
    27500  4.00   1.000      -      -  done      T27500_G4.00_Z1.000_V10.tar.gz -> T27500_G4.00_Z1.000_V10.tar.gz
    30000  3.00   1.000      -      -  renamed   BG30000g300v2.tar.gz -> T30000_G3.00_Z1.000_V10.tar.gz
    30000  3.00   1.000      -      -  skipped   G30000g300v10.tar.gz -> T30000_G3.00_Z1.000_V10.tar.gz  [same target as BG30000g300v2.tar.gz]
    30000  4.00   1.000      -      -  renamed   G30000g400v10.tar.gz -> T30000_G4.00_Z1.000_V10.tar.gz
    35000  4.00   2.000      -      -  error     C35000g400v10.tar.gz -> T3